# MLOps 

This notebook is intended to help you get started managing your machine learning models and deployments in the H2O AI Cloud using python. 

**_This notebook assumes you have run 1 End to End Demo and have a workspace with a deployed model that exists with a specific name_**

* **Product Documentation:** https://docs.h2o.ai/mlops/
* **Python Documentation:** https://docs.h2o.ai/mlops/py-client/overview


## Prerequistes

Update the `h2o_ai_cloud.py` file with the connection parameters for your H2O AI Cloud environemnt:
1. Login to your H2O AI Cloud environment
1. Click your username or avatar in the H2O AI Cloud navigation bar
1. Navigate to `CLI & API Access`
1. Use the variables from the `Accessing H2O AI Cloud APIs` section to populate the parameters

In [1]:
import requests
import json

from h2o_ai_cloud import mlops_client

import pandas as pd

## Securely connect to the platform
We first connect to the H2O AI Cloud using our platform token to create a token provider object. We can then use this object to log into AI Engine Manager and other APIs.

## Connect to MLOps

In [2]:
mlops = mlops_client()

Visit https://internal.dedicated.h2o.ai/auth/get-platform-token to get your platform token


## Workspaces

### Create a Workspace

The workspace is a place to compile a workspace with all its models, models versions and deployments. It is linked to the concept of "workspace" you link experiments to in DriverlessAI.

In [3]:
workspace = mlops.workspaces.create(name="my_test_workspace", description="My test workspace for H2O MLOps tutorial")

### List all Workspaces you have access to (shared or owned)

In [ ]:
my_workspaces = mlops.workspaces.list()

for w in my_workspaces:
    print(w.uid, w.name)

### Delete a workspace 

In [7]:
for w in my_workspaces:
    if w.name == "my_test_workspace":
        w.delete()

In [ ]:
my_workspaces = mlops.workspaces.list()

for w in my_workspaces:
    print(w.uid, w.name)

### Select a specific workspace to work with
We have previously created this workspace using Driverless AI and added models to it in our first tutorial.

In [9]:
for w in my_workspaces:
    if w.name == "Telco Churn Predictions":
        workspace_id = w.uid
        workspace = w

## Experiments

### List experiments
Get a list of all experiments that are in our workspace (added to that workspace via DriverlessAI Experiments linking - please refer to `3 DriverlessAI`)

In [11]:
my_workspace_experiments = workspace.experiments.list()
my_workspace_experiments

    | name             | uid                                  | tags
----+------------------+--------------------------------------+--------
  0 | Default Baseline | db5939ce-b6a0-11f1-977a-f2eab8d67387 |

In [12]:
my_workspace_experiments = workspace.experiments.list()

for exp in my_workspace_experiments:
    print(exp.uid, exp.name)

db5939ce-b6a0-11f1-977a-f2eab8d67387 Default Baseline


### Select an experiment

In [14]:
experiment = my_workspace_experiments[0]

experiment

<class 'h2o_mlops._experiments.MLOpsExperiment(
    uid='db5939ce-b6a0-11f1-977a-f2eab8d67387',
    name='Default Baseline',
    description='',
    creator_uid='c19820dc-37ef-4d4f-893d-5f46492e51a0',
    created_time=datetime.datetime(2026, 9, 22, 16, 18, 58, 459647, tzinfo=TzInfo(0)),
    last_modified_time=datetime.datetime(2026, 9, 22, 16, 18, 58, 459647, tzinfo=TzInfo(0)),
)'>

In [18]:
experiment_artefacts = experiment.artifacts.list()

In [36]:
experiment.scoring_runtimes

    | artifact_type               | runtime_uid                                                | runtime_name
----+-----------------------------+------------------------------------------------------------+------------------------------------
  0 | dai_python_scoring_pipeline | workspaces/global/runtimes/python-scorer_dai_pipelines_250 | Python Pipeline Scorer [DAI 2.5.0]
  1 | dai_mojo_pipeline           | workspaces/global/runtimes/dai_mojo_scorer                 | DAI MOJO Scorer

Select Mojo scoring runtime if available (as faster), otherwise select Python scoring runtime:

In [38]:
for runtime in experiment.scoring_runtimes:
    if runtime.artifact_type == "dai_mojo_pipeline":
        scoring_runtime = runtime
    else:
        scoring_runtime = runtime

scoring_runtime

<class 'h2o_mlops._runtimes.MLOpsScoringRuntime(
    runtime='workspaces/global/runtimes/dai_mojo_scorer',
    artifact_type='dai_mojo_pipeline',
    artifact_processor='dai_mojo_pipeline_extractor',
    model_type='dai_mojo',
)'>

## Deployments

In [35]:
import h2o_mlops.options as options
import h2o_mlops.types as types

In [ ]:
registered_model_name = "Telco Churn Predictions"

secure_endpoint_password = < secure_endpoint_password >
static_endpoint = "telco_churn_predictions"

In [47]:
workspace.deployments.list()

    | name   | mode   | uid
----+--------+--------+-------

In [53]:
model = workspace.models.list(name=registered_model_name)[0]

In [ ]:
deployment = workspace.deployments.create(
   name="my-deployment",
#    model = workspace.models.list(name=registered_model_name)[0],
   composition_options=options.CompositionOptions(
       model=workspace.models.list(name =registered_model_name)[0],
       scoring_runtime=scoring_runtime,
   ),

   kubernetes_options=options.KubernetesOptions(
               requests = {'cpu': '200m', 'memory': '130Mi'},
               ),

    security_options=options.SecurityOptions(
            security_type=types.SecurityType.HASHED_PASSPHRASE, # types.SecurityType.DISABLED,
            passphrase=secure_endpoint_password,
    ),
    
    monitoring_options = None, # Check out https://docs.h2o.ai/mlops/py-client/examples/monitoring-setup for options

    #mode = "AB_TEST", # DEFAULT - SINGLE_MODEL Other Options: CHAMPION_CHALLENGER, AB_TEST, or SHADOW. - expect a list of two CompositionOptions objects for multi-model modes.
)

In [58]:
workspace.deployments.list()

    | name          | mode                            | uid
----+---------------+---------------------------------+--------------------------------------
  0 | my-deployment | DeploymentModeType.SINGLE_MODEL | be6d46c5-9b6a-4746-9d5a-edc8d664ace5

In [88]:
deployment.wait_for_healthy()

Setting a static endpoint for the deployment allows you to have a fixed URL for your model, which can be useful for integration 
with other systems or for easier access. You can set a static endpoint by specifying the `static_endpoint` parameter when creating the deployment.
    

In [ ]:
endpoint = workspace.endpoints.list(name=static_endpoint)

if not endpoint:
    endpoint = deployment.configure_endpoint(
        path=static_endpoint,
        )
else:
    # if Static Endpoint exists already
    endpoint = endpoint[0]
    
    
#Attaching deployment with static endpoint
endpoint.update(target_deployment=deployment)

## Make predictions

### Display Deployment Scorer Capabilities

In [124]:
deployment.scorer.capabilities

<bound method MLOpsDeploymentScorer.capabilities of <class 'h2o_mlops._deployments.MLOpsDeploymentScorer(
    api_base_url='https://model.internal.dedicated.h2o.ai/b750aaed-4e27-465a-b5f4-da08c77903f3',
    capabilities_endpoint='https://model.internal.dedicated.h2o.ai/b750aaed-4e27-465a-b5f4-da08c77903f3/model/capabilities',
    schema_endpoint='https://model.internal.dedicated.h2o.ai/b750aaed-4e27-465a-b5f4-da08c77903f3/model/schema',
    sample_request_endpoint='https://model.internal.dedicated.h2o.ai/b750aaed-4e27-465a-b5f4-da08c77903f3/model/sample_request',
    scoring_endpoint='https://model.internal.dedicated.h2o.ai/b750aaed-4e27-465a-b5f4-da08c77903f3/model/score',
)'>>

### Get Sample request example to score

In [125]:
scorer = deployment.scorer
sample_request = scorer.sample_request(auth_value=secure_endpoint_password)
print("Sample request:", sample_request)

Sample request: {'fields': ['State', 'Account Length', 'Area Code', "Int'l Plan", 'VMail Plan', 'VMail Message', 'Day Mins', 'Day Calls', 'Day Charge', 'Eve Mins', 'Eve Calls', 'Eve Charge', 'Night Mins', 'Night Calls', 'Night Charge', 'Intl Mins', 'Intl Calls', 'Intl Charge', 'CustServ Calls'], 'rows': [['State', '1.0', '1.0', "Int'l Plan", 'VMail Plan', '1.0', '1.0', '1.0', '1.0', '1.0', '1.0', '1.0', '1.0', '1.0', '1.0', '1.0', '1.0', '1.0', '1.0']]}


In [126]:
fields = sample_request["fields"]
fields

['State',
 'Account Length',
 'Area Code',
 "Int'l Plan",
 'VMail Plan',
 'VMail Message',
 'Day Mins',
 'Day Calls',
 'Day Charge',
 'Eve Mins',
 'Eve Calls',
 'Eve Charge',
 'Night Mins',
 'Night Calls',
 'Night Charge',
 'Intl Mins',
 'Intl Calls',
 'Intl Charge',
 'CustServ Calls']

In [127]:
sample_row = sample_request["rows"][0]
sample_row

['State',
 '1.0',
 '1.0',
 "Int'l Plan",
 'VMail Plan',
 '1.0',
 '1.0',
 '1.0',
 '1.0',
 '1.0',
 '1.0',
 '1.0',
 '1.0',
 '1.0',
 '1.0',
 '1.0',
 '1.0',
 '1.0',
 '1.0']

#### Use the static endpoint to score data on the deployment 

In [129]:
predictions = requests.post(
    url = f"https://model.internal.dedicated.h2o.ai/{static_endpoint}/model/score",
    json={
        'fields': fields,
        'rows': [sample_row]
    },
    headers = {"Content-Type": "application/json",
               "Authorization": f"Bearer {secure_endpoint_password}"}
)

predictions = predictions.json()

In [130]:
pd.DataFrame(predictions["score"], columns=predictions["fields"])

,Churn?.False.,Churn?.True.
0,0.9543405401470918,0.04565945985290816


#### Use H2O MLOps client directly (deployment scorer object)

In [131]:
deployment.scorer.score(sample_request, auth_value=secure_endpoint_password)

{'id': 'db5939ce-b6a0-11f1-977a-f2eab8d67387',
 'fields': ['Churn?.False.', 'Churn?.True.'],
 'score': [['0.9543405401470918', '0.04565945985290816']]}

## Delete a deployment

In [132]:
deployment.delete()